# Brute-force phase-center calculation from HFSS `rETheta`

This notebook calculates the E-plane phase center from HFSS far-field complex data.

Assumptions:

- Horn boresight: `+z`
- TE11 co-polar electric field: `x`
- E-plane: `Phi = 0 deg`
- Complex field used: `rETheta = re(rETheta) + 1j * im(rETheta)`
- For each frequency, the phase center is searched along the `z` axis.
- For a candidate position \(z\),

\[
\psi_z(\theta,f)
=
\psi_0(\theta,f)
+
s\,k(f)z\cos\theta,
\]

where \(s=\pm1\) is controlled by `PHASE_SIGN`.

The objective is

\[
\sigma_\psi(z,f)
=
\operatorname{std}_{\theta}
\left[\psi_z(\theta,f)\right],
\]

and

\[
z_{\rm pc}(f)
=
\arg\min_z \sigma_\psi(z,f).
\]

The final broadband stability metric is

\[
\operatorname{std}_f[z_{\rm pc}(f)].
\]

The HFSS export format assumed here is a wide table:

- first column: `Theta [deg]`
- remaining columns: one frequency per column
- one file for `re(rETheta)`
- one file for `im(rETheta)`

The Re and Im files must contain the same theta samples and frequencies.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

C0 = 299_792_458.0

## 1. User settings

Set the paths to the two HFSS exports.

The attached example Re file has headers such as:

`re(rETheta) [V] - Freq='80GHz' Phi='0deg'`

The Im file should have the corresponding form:

`im(rETheta) [V] - Freq='80GHz' Phi='0deg'`

In [ ]:
# ---- Input files ----
RE_FILE = Path("re_rETheta.csv")
IM_FILE = Path("im_rETheta.csv")

# ---- Analysis range ----
THETA_MIN_DEG = -10.0
THETA_MAX_DEG = +10.0

# ---- z brute-force search ----
Z_MIN_MM = -30.0
Z_MAX_MM = +30.0
DZ_MM = 0.02

# +1 corresponds to:
# phase_z = phase0 + k*z*cos(theta)
# If the HFSS phase convention / z-axis definition gives the opposite
# physical sign, change this to -1.
PHASE_SIGN = +1

# Diagnostic frequency. If None, use the middle available frequency.
DIAGNOSTIC_FREQ_GHZ = 100.0

# Result CSV
OUTPUT_CSV = Path("phase_center_results.csv")

## 2. Load the HFSS wide tables

The frequency is parsed directly from each column name, so the code does not assume a fixed 80–180 GHz grid.

In [ ]:
FREQ_RE = re.compile(r"Freq='([0-9.]+)GHz'")

def load_hfss_wide(path, component):
    df = pd.read_csv(path)

    theta_col = df.columns[0]
    theta = df[theta_col].to_numpy(dtype=float)

    data = {}
    for col in df.columns[1:]:
        match = FREQ_RE.search(col)
        if match is None:
            continue

        freq_ghz = float(match.group(1))

        if component.lower() == "re" and "re(rETheta)" not in col:
            continue
        if component.lower() == "im" and "im(rETheta)" not in col:
            continue

        data[freq_ghz] = df[col].to_numpy(dtype=float)

    if not data:
        raise ValueError(
            f"No {component}(rETheta) frequency columns were found in {path}"
        )

    return theta, data


theta_re_deg, re_data = load_hfss_wide(RE_FILE, "re")
theta_im_deg, im_data = load_hfss_wide(IM_FILE, "im")

if len(theta_re_deg) != len(theta_im_deg) or not np.allclose(theta_re_deg, theta_im_deg):
    raise ValueError("Theta samples in Re and Im files do not match.")

theta_deg_all = theta_re_deg

frequencies_ghz = np.array(
    sorted(set(re_data.keys()) & set(im_data.keys())),
    dtype=float,
)

if len(frequencies_ghz) == 0:
    raise ValueError("No common frequencies were found in the Re and Im files.")

print(f"Theta samples: {len(theta_deg_all)}")
print(
    f"Common frequencies: {frequencies_ghz[0]:g} "
    f"to {frequencies_ghz[-1]:g} GHz "
    f"({len(frequencies_ghz)} points)"
)

## 3. Restrict the calculation to the E-plane main-beam theta range

The phase is unwrapped after sorting by theta.

In [ ]:
theta_mask = (
    (theta_deg_all >= THETA_MIN_DEG)
    & (theta_deg_all <= THETA_MAX_DEG)
)

theta_deg = theta_deg_all[theta_mask]

if len(theta_deg) < 3:
    raise ValueError(
        "Too few theta samples in the selected main-beam range. "
        "Check THETA_MIN_DEG / THETA_MAX_DEG and the HFSS export."
    )

order = np.argsort(theta_deg)
theta_deg = theta_deg[order]
theta_rad = np.deg2rad(theta_deg)

z_candidates_mm = np.arange(
    Z_MIN_MM,
    Z_MAX_MM + 0.5 * DZ_MM,
    DZ_MM,
)
z_candidates_m = z_candidates_mm * 1e-3

print(
    f"Theta range used: {theta_deg[0]:g} to {theta_deg[-1]:g} deg "
    f"({len(theta_deg)} samples)"
)
print(
    f"z search: {z_candidates_mm[0]:g} to {z_candidates_mm[-1]:g} mm "
    f"with dz = {DZ_MM:g} mm"
)

## 4. Phase-center solver for one frequency

For one frequency:

1. Construct \(E_\theta(\theta)=\Re E_\theta+i\Im E_\theta\).
2. Calculate and unwrap the original phase.
3. Sweep candidate \(z\).
4. For each \(z\), calculate phase over all theta samples.
5. Calculate `std(phase)` over theta.
6. Take the `argmin` over \(z\).

The theta sweep is vectorized with NumPy.

In [ ]:
def phase_center_at_frequency(freq_ghz):
    re_e = re_data[freq_ghz][theta_mask][order]
    im_e = im_data[freq_ghz][theta_mask][order]

    e_theta = re_e + 1j * im_e
    phase0 = np.unwrap(np.angle(e_theta))

    freq_hz = freq_ghz * 1e9
    k = 2.0 * np.pi * freq_hz / C0

    # Shape:
    #   z_candidates_m[:, None] -> (Nz, 1)
    #   theta_rad[None, :]       -> (1, Ntheta)
    # Result:
    #   phase_z                 -> (Nz, Ntheta)
    phase_z = (
        phase0[None, :]
        + PHASE_SIGN
        * k
        * z_candidates_m[:, None]
        * np.cos(theta_rad)[None, :]
    )

    phase_std = np.std(phase_z, axis=1)

    best_index = int(np.argmin(phase_std))
    z_pc_m = z_candidates_m[best_index]

    return {
        "freq_ghz": freq_ghz,
        "z_pc_m": z_pc_m,
        "z_pc_mm": z_pc_m * 1e3,
        "min_phase_std_rad": phase_std[best_index],
        "min_phase_std_deg": np.rad2deg(phase_std[best_index]),
        "phase0": phase0,
        "phase_best": phase_z[best_index],
        "phase_std_vs_z": phase_std,
        "best_index": best_index,
    }

## 5. Calculate the phase center at every frequency

In [ ]:
results = []

for freq_ghz in frequencies_ghz:
    result = phase_center_at_frequency(freq_ghz)

    results.append(
        {
            "Frequency [GHz]": result["freq_ghz"],
            "Phase center z [mm]": result["z_pc_mm"],
            "Minimum phase STD [rad]": result["min_phase_std_rad"],
            "Minimum phase STD [deg]": result["min_phase_std_deg"],
        }
    )

results_df = pd.DataFrame(results)

mean_z_pc_mm = results_df["Phase center z [mm]"].mean()
std_z_pc_mm = results_df["Phase center z [mm]"].std(ddof=0)

print(results_df.to_string(index=False))
print()
print(f"Mean phase center = {mean_z_pc_mm:.4f} mm")
print(f"Phase-center STD over frequency = {std_z_pc_mm:.4f} mm")

results_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

## 6. Plot phase center versus frequency

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    results_df["Frequency [GHz]"],
    results_df["Phase center z [mm]"],
    marker="o",
    markersize=3,
)

ax.axhline(
    mean_z_pc_mm,
    linestyle="--",
    label=f"Mean = {mean_z_pc_mm:.3f} mm",
)

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Phase center z [mm]")
ax.grid(True)
ax.legend()

plt.show()

## 7. Diagnostic plot at one frequency

This plot shows:

- the brute-force objective `phase STD vs z`
- the original phase versus theta
- the phase versus theta after moving the reference origin to the best \(z\)

This is useful for checking that the minimum is not sitting at the edge of the selected `z` search range.

In [ ]:
if DIAGNOSTIC_FREQ_GHZ is None:
    diagnostic_freq_ghz = frequencies_ghz[len(frequencies_ghz) // 2]
else:
    diagnostic_freq_ghz = frequencies_ghz[
        np.argmin(np.abs(frequencies_ghz - DIAGNOSTIC_FREQ_GHZ))
    ]

diag = phase_center_at_frequency(float(diagnostic_freq_ghz))

print(
    f"Diagnostic frequency: {diag['freq_ghz']:g} GHz\n"
    f"z_pc = {diag['z_pc_mm']:.4f} mm\n"
    f"minimum phase STD = {diag['min_phase_std_deg']:.4f} deg"
)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    z_candidates_mm,
    np.rad2deg(diag["phase_std_vs_z"]),
)

ax.axvline(
    diag["z_pc_mm"],
    linestyle="--",
    label=f"z_pc = {diag['z_pc_mm']:.3f} mm",
)

ax.set_xlabel("Candidate z [mm]")
ax.set_ylabel("Phase STD over theta [deg]")
ax.grid(True)
ax.legend()

plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    theta_deg,
    np.rad2deg(diag["phase0"]),
    label="Original HFSS origin",
)

ax.plot(
    theta_deg,
    np.rad2deg(diag["phase_best"]),
    label=f"Shifted to z = {diag['z_pc_mm']:.3f} mm",
)

ax.set_xlabel("Theta [deg]")
ax.set_ylabel("Unwrapped phase [deg]")
ax.grid(True)
ax.legend()

plt.show()

## Notes

- If the optimum always appears at `Z_MIN_MM` or `Z_MAX_MM`, widen the z search range.
- `PHASE_SIGN` controls only the reported physical sign of the phase-center position. If needed, compare with an HFSS reference-origin shift to determine the convention.
- The absolute phase offset is irrelevant because the objective is the standard deviation over theta.
- `std_f(z_pc)` measures broadband phase-center stability. The absolute mean phase-center position is mainly relevant to integration with the rest of the optical system.